In [0]:
# ============================================================
# PRODUCTS SILVER
# Bronze → Silver
# ============================================================

from pyspark.sql.functions import (
    col,
    trim,
    initcap,
    row_number,
    current_timestamp,
    lit
)

from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
# ============================================================
# 1. PATHS
# ============================================================

bronze_path = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "bronze/products/"
)

silver_path = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "silver/products/"
)

quarantine_path = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "quarantine/products/"
)

control_path = (
    "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/"
    "metadata/pipeline_watermark/"
)

pipeline_name = "products_silver"

In [0]:
# ============================================================
# 2. READ BRONZE
# ============================================================

products_bronze_df = (
    spark.read
    .format("delta")
    .load(bronze_path)
)

print("Bronze records:", products_bronze_df.count())
display(products_bronze_df.select("*").limit(100))

# ============================================================
# 3. READ SHARED WATERMARK TABLE
# ============================================================

control_df = (
    spark.read
    .format("delta")
    .load(control_path)
)
display(control_df)
print("Shared control table loaded.")

Bronze records: 125003


product_id,product_name,category,brand,price,_source_file,_source_path,_dataset,_batch_id,_run_id,_ingested_at,_source_system
230001,Product Oct7 Updated 1,Beverages,BrandA,149.99,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,3eaae59a-bdb5-4fb0-a99a-2f61842ed11b,fca4af74-ff0e-4b2c-8d4a-fdd143087a4b,2026-10-12T13:08:37.434Z,product_api
230002,Product Oct7 Updated 2,Snacks,BrandB,219.49,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,3eaae59a-bdb5-4fb0-a99a-2f61842ed11b,fca4af74-ff0e-4b2c-8d4a-fdd143087a4b,2026-10-12T13:08:37.434Z,product_api
230003,Product Oct7 Updated 3,Nutrition,BrandC,329.75,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,3eaae59a-bdb5-4fb0-a99a-2f61842ed11b,fca4af74-ff0e-4b2c-8d4a-fdd143087a4b,2026-10-12T13:08:37.434Z,product_api
230004,Product Oct7 Updated 4,Cereals,BrandD,189.25,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,3eaae59a-bdb5-4fb0-a99a-2f61842ed11b,fca4af74-ff0e-4b2c-8d4a-fdd143087a4b,2026-10-12T13:08:37.434Z,product_api
230005,Product Oct7 Updated 5,Dairy,BrandE,279.9,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,3eaae59a-bdb5-4fb0-a99a-2f61842ed11b,fca4af74-ff0e-4b2c-8d4a-fdd143087a4b,2026-10-12T13:08:37.434Z,product_api
230006,Product Oct7 Updated 6,Beverages,BrandB,119.5,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,3eaae59a-bdb5-4fb0-a99a-2f61842ed11b,fca4af74-ff0e-4b2c-8d4a-fdd143087a4b,2026-10-12T13:08:37.434Z,product_api
260007,Product Oct7 7,Cereals,BrandE,76.59,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,3eaae59a-bdb5-4fb0-a99a-2f61842ed11b,fca4af74-ff0e-4b2c-8d4a-fdd143087a4b,2026-10-12T13:08:37.434Z,product_api
260008,Product Oct7 8,Dairy,BrandA,83.96,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,3eaae59a-bdb5-4fb0-a99a-2f61842ed11b,fca4af74-ff0e-4b2c-8d4a-fdd143087a4b,2026-10-12T13:08:37.434Z,product_api
260009,Product Oct7 9,Beverages,BrandB,91.33,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,3eaae59a-bdb5-4fb0-a99a-2f61842ed11b,fca4af74-ff0e-4b2c-8d4a-fdd143087a4b,2026-10-12T13:08:37.434Z,product_api
260010,Product Oct7 10,Snacks,BrandC,98.7,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,products,3eaae59a-bdb5-4fb0-a99a-2f61842ed11b,fca4af74-ff0e-4b2c-8d4a-fdd143087a4b,2026-10-12T13:08:37.434Z,product_api


pipeline_name,last_processed_timestamp,last_run_id,updated_at
customers_silver,2026-10-14T08:35:16.583Z,"CUSTOMERS_REBUILD_Column<'date_format(current_timestamp(), yyyyMMddHHmmss)'>",2026-10-14T08:35:47.290Z
gold_product_sales,2026-10-08T17:31:58.735Z,GOLD_PRODUCT_SALES_20261008173158,2026-10-08T17:33:22.448Z
gold_daily_sales,2026-10-08T17:31:58.735Z,GOLD_DAILY_SALES_20261008173158,2026-10-08T17:33:10.714Z
gold_customer_sales,2026-10-08T17:31:58.735Z,GOLD_CUSTOMER_SALES_RUN,2026-10-11T15:31:33.202Z
gold_market_sales,2026-10-08T17:31:58.735Z,GOLD_MARKET_SALES_RUN,2026-10-08T17:33:19.347Z
orders_silver,2026-10-09T02:49:13.000Z,RUN_49e099d8,2026-10-08T17:32:40.886Z
products_silver,2026-10-14T08:35:15.240Z,null,2026-10-14T12:53:52.608Z


Shared control table loaded.


In [0]:
# ============================================================
# 4. GET PRODUCTS WATERMARK
# ============================================================
pipeline_row = (
    control_df
    .filter(col("pipeline_name") == pipeline_name)
)
if pipeline_row.count() == 0:
    print("Products pipeline row does not exist.")
    new_row = spark.createDataFrame(
        [
            (
                pipeline_name,
                None,
                None,
                None
            )
        ],
        control_df.schema
    )
    (
        new_row
        .write
        .format("delta")
        .mode("append")
        .save(control_path)
    )
    last_processed_timestamp = None
    print("Products pipeline row created.")
else:
    last_processed_timestamp = (
        pipeline_row
        .select("last_processed_timestamp")
        .first()[0]
    )
    print(
       "Last processed timestamp:",
        last_processed_timestamp
    )

Last processed timestamp: 2026-10-14 08:35:15.240942


In [0]:
# ============================================================
# 5. INCREMENTAL DATA
# ============================================================
if last_processed_timestamp is None:
    products_incremental_df = products_bronze_df
    print("First Products Silver run.")
    print("Processing all Bronze records.")
else:
    products_incremental_df = (
        products_bronze_df
        .filter(
            col("_ingested_at") > last_processed_timestamp
        )
    )
    print("Incremental Products run.")
incremental_count = products_incremental_df.count()
print(
    "Incremental records:",
    incremental_count
)

Incremental Products run.
Incremental records: 0


In [0]:

# ============================================================
# 6. STOP IF NOTHING NEW
# ============================================================

if incremental_count == 0:

    print("No new Products records.")
    print("Products Silver is already up to date.")

else:

    # ========================================================
    # 7. CLEAN COLUMNS
    # ========================================================

    products_clean_df = (
        products_incremental_df

        .withColumn(
            "product_id",
            col("product_id").cast("long")
        )

        .withColumn(
            "product_name",
            initcap(trim(col("product_name")))
        )

        .withColumn(
            "category",
            initcap(trim(col("category")))
        )

        .withColumn(
            "brand",
            initcap(trim(col("brand")))
        )

        .withColumn(
            "price",
            col("price").cast("double")
        )
    )
    display(products_clean_df.count())

    # ========================================================
    # 8. EXACT DUPLICATE REMOVAL
    # ========================================================

    print(
        "Before exact dedup:",
        products_clean_df.count()
    )

    products_clean_df = (
        products_clean_df
        .dropDuplicates()
    )

    print(
        "After exact dedup:",
        products_clean_df.count()
    )


    # ========================================================
    # 9. PRODUCT_ID DEDUPLICATION
    # ========================================================
    # Keep the latest record for each product_id

    product_window = (
        Window
        .partitionBy("product_id")
        .orderBy(
            col("_ingested_at").desc()
        )
    )

    products_dedup_df = (
        products_clean_df
        .withColumn(
            "row_number",
            row_number().over(product_window)
        )
        .filter(
            col("row_number") == 1
        )
        .drop("row_number")
    )

    print(
        "After product_id dedup:",
        products_dedup_df.count()
    )


    # ========================================================
    # 10. DATA QUALITY VALIDATION
    # ========================================================


    invalid_condition = (
        col("product_id").isNull()
        |
        col("product_name").isNull()
        |
        col("category").isNull()
        |
        col("brand").isNull()
        |
        col("price").isNull()
        |
        (col("price") <= 0)

    )

    invalid_products_df = (
        products_dedup_df
        .filter(invalid_condition)
    )

    valid_products_df = (
        products_dedup_df
        .filter(~invalid_condition)
    )

    valid_count = valid_products_df.count()
    invalid_count = invalid_products_df.count()

    print(
        "Valid Products:",
        valid_count
    )

    print(
        "Invalid Products:",
        invalid_count
    )


    # ========================================================
    # 11. QUARANTINE INVALID RECORDS
    # ========================================================

    if invalid_count > 0:

        (
            invalid_products_df
            .write
            .format("delta")
            .mode("append")
            .save(quarantine_path)
        )

        print(
            "Invalid Products written to quarantine."
        )

    else:

        print(
            "No invalid Products."
        )


    # ========================================================
    # 12. CREATE PRODUCTS SILVER
    # ========================================================

    try:

        dbutils.fs.ls(
            silver_path + "_delta_log/"
        )

        silver_exists = True

    except Exception:

        silver_exists = False


    if not silver_exists:

        print(
            "Products Silver does not exist."
        )

        print(
            "Creating Products Silver..."
        )

        (
            valid_products_df
            .write
            .format("delta")
            .mode("overwrite")
            .save(silver_path)
        )

        print(
            "Products Silver created successfully."
        )


# display(spark.read.format("delta").load(silver_path))
    # ========================================================
    # 13. MERGE INTO EXISTING SILVER
    # ========================================================
    else:
        print(
            "Products Silver already exists."
        )
        products_silver_table = (
            DeltaTable.forPath(
                spark,
                silver_path
            )
        )
        (
    products_silver_table.alias("target")
    .merge(
        valid_products_df.alias("source"),
        "target.product_id = source.product_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
        print(
            "Products Silver MERGE completed."
        )

    # ========================================================
    # 14. UPDATE WATERMARK
    # ========================================================

    if valid_count > 0:

        new_watermark = (
          valid_products_df
         .agg({
            "_ingested_at": "max"
          })
          .first()[0]
         )

        print(
            "New watermark:",
            new_watermark
        )

        control_table = (
            DeltaTable.forPath(
                spark,
                control_path
            )
        )

        watermark_df = spark.createDataFrame(
            [
                (
                    pipeline_name,
                    new_watermark
                )
            ],
            [
                "pipeline_name",
                "last_processed_timestamp"
            ]
        )

        (
            control_table.alias("target")
            .merge(
                watermark_df.alias("source"),
                "target.pipeline_name = source.pipeline_name"
            )

            .whenMatchedUpdate(
                set={
                    "last_processed_timestamp":
                        "source.last_processed_timestamp",

                    "updated_at":
                        "current_timestamp()"
                }
            )

            .whenNotMatchedInsert(
                values={
                    "pipeline_name":
                        "source.pipeline_name",

                    "last_processed_timestamp":
                        "source.last_processed_timestamp",

                    "last_run_id":
                        "NULL",

                    "updated_at":
                        "current_timestamp()"
                }
            )

            .execute()
        )

        print(
            "Watermark updated successfully."
        )

    else:

        print(
            "No valid Products. Watermark not updated."
        )




No new Products records.
Products Silver is already up to date.


In [0]:
silver_df = spark.read.format("delta").load(silver_path)
print("Total Silver rows:", silver_df.count())
print(
    "Distinct product IDs:",
    silver_df.select("product_id").distinct().count()
)
print(
    "Duplicate product IDs:",
    silver_df.count()
    - silver_df.select("product_id").distinct().count()
)

Total Silver rows: 120003
Distinct product IDs: 120003
Duplicate product IDs: 0


In [0]:
# ============================================================
# 15. FINAL VERIFICATION
# ============================================================

print("\n========== FINAL PRODUCTS SILVER CHECK ==========")

try:

    products_silver_df = (
        spark.read
        .format("delta")
        .load(silver_path)
    )

    print(
        "Products Silver records:",
        products_silver_df.count()
    )

    display(products_silver_df.select("*").limit(100))

except Exception as e:

    print(
        "Products Silver verification failed."
    )

    print(e)


========== FINAL PRODUCTS SILVER CHECK ==========
Products Silver records: 120003


product_id,product_name,category,brand,price,_source_file,_source_path,_dataset,_batch_id,_run_id,_ingested_at,_source_system
230009,Usb-c Hub,Accessories,Anker,1499.0,products_20261012.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=12/products_20261012.json,products,cb8f6ca4-2013-4c07-845e-b4846bce0780,d004e230-d235-4f9e-a1a4-063d81832361,2026-10-14T08:31:33.169Z,product_api
240009,Product 240009,Snacks,Kitkat,199.48,products_20261014.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=14/products_20261014.json,products,8353ddc9-f1ac-4bf1-a0d2-4f028e36f8bf,b90d11f8-e74f-4776-b3f2-5fe9948f8910,2026-10-14T08:35:15.240Z,product_api
240075,Product 240075,Snacks,Kitkat,61.32,products_20261014.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=14/products_20261014.json,products,8353ddc9-f1ac-4bf1-a0d2-4f028e36f8bf,b90d11f8-e74f-4776-b3f2-5fe9948f8910,2026-10-14T08:35:15.240Z,product_api
240077,Product 240077,Personal Care,Purina,91.98,products_20261014.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=14/products_20261014.json,products,8353ddc9-f1ac-4bf1-a0d2-4f028e36f8bf,b90d11f8-e74f-4776-b3f2-5fe9948f8910,2026-10-14T08:35:15.240Z,product_api
240089,Product 240089,Personal Care,Purina,175.73,products_20261014.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=14/products_20261014.json,products,8353ddc9-f1ac-4bf1-a0d2-4f028e36f8bf,b90d11f8-e74f-4776-b3f2-5fe9948f8910,2026-10-14T08:35:15.240Z,product_api
240091,Product 240091,Beverages,Nestle,218.75,products_20261014.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=14/products_20261014.json,products,8353ddc9-f1ac-4bf1-a0d2-4f028e36f8bf,b90d11f8-e74f-4776-b3f2-5fe9948f8910,2026-10-14T08:35:15.240Z,product_api
240096,Product 240096,Pet Care,Nan,201.13,products_20261014.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=14/products_20261014.json,products,8353ddc9-f1ac-4bf1-a0d2-4f028e36f8bf,b90d11f8-e74f-4776-b3f2-5fe9948f8910,2026-10-14T08:35:15.240Z,product_api
240110,Product 240110,Dairy,Nescafe,98.73,products_20261014.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=14/products_20261014.json,products,8353ddc9-f1ac-4bf1-a0d2-4f028e36f8bf,b90d11f8-e74f-4776-b3f2-5fe9948f8910,2026-10-14T08:35:15.240Z,product_api
240123,Product 240123,Snacks,Kitkat,109.47,products_20261014.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=14/products_20261014.json,products,8353ddc9-f1ac-4bf1-a0d2-4f028e36f8bf,b90d11f8-e74f-4776-b3f2-5fe9948f8910,2026-10-14T08:35:15.240Z,product_api
240125,Product 240125,Personal Care,Purina,138.75,products_20261014.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=14/products_20261014.json,products,8353ddc9-f1ac-4bf1-a0d2-4f028e36f8bf,b90d11f8-e74f-4776-b3f2-5fe9948f8910,2026-10-14T08:35:15.240Z,product_api
